# Approche 5 — `VirtualAddressRecurrentCoupler` (Item 5)

Cinquième notebook d'attention pour l'équipe R&D RTE. Il déroule l'Item 5 (état virtuel partagé) sur le serveur GPU de La Javaness.

**Prérequis :** notebooks 00 à 04 lus au préalable.

**Durée attendue :** 7 à 12 minutes sur le serveur GPU de La Javaness.


## 1. Mise en place de l'environnement

Imports : JAX, NumPy, Flax NNX, optax. Vérification du device JAX et du dtype par défaut. On attend `CudaDevice` (serveur GPU de La Javaness) et `float32`.


In [1]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root / "benchmarks"))

print(f"JAX version:    {jax.__version__}")
print(f"JAX devices:    {jax.devices()}")
print(f"Default dtype:  {jnp.array(0.0).dtype}")
print(f"Repo root:      {_root}")


JAX version:    0.9.0


JAX devices:    [CudaDevice(id=0), CudaDevice(id=1)]
Default dtype:  float32
Repo root:      /home/van.tuan/GNN/energnn_attention


## 2. Méthode — `VirtualAddressRecurrentCoupler`

L'Item 5 du backlog (section 3.5) propose un coupler qui étend `RecurrentCoupler` avec un état virtuel partagé `h_virtual` évoluant en parallèle de l'état per-adresse `h`. Le scaffold amont laissait deux TODO à instancier (injection de `h_virtual` et définition de `F_virtual`). Les choix retenus sont les suivants&nbsp;:

**Injection (design (c))**. `h_virtual` est concaténé au vecteur de messages avant `phi`, pas avant les message functions. Aucune modification de l'ABC `MessagePassingFunction.__call__`. Les Items 1 à 4 sont réutilisés tels quels.

**F_virtual (v1 livrée)**. `F_virtual = phi_virtual(concat(masked_mean(h), h_virtual_old))`. La moyenne masquée utilise le dénominateur corrigé `sum(non_fictitious_addresses) + eps`, identique à l'Approche 2 (`GlobalAggregationMessagePassingFunction`).

**Adresse virtuelle unique**. Un seul vecteur `h_virtual ∈ ℝ^{d_v}`. L'extension à plusieurs adresses virtuelles n'est pas couverte ici.

Forme du forward Euler joint à chaque pas&nbsp;:

$$h(t+\delta t) = h(t) + \delta t \cdot \phi\!\left( \text{concat}\big(\psi_1(h), ..., \psi_n(h), h_{\text{virtual}}(t) \otimes \mathbf{1}_{n_{\text{addr}}}\big) \right)$$

$$h_{\text{virtual}}(t+\delta t) = h_{\text{virtual}}(t) + \delta t \cdot \phi_{\text{virtual}}\!\left( \text{concat}\big(\text{mean}(h(t)), h_{\text{virtual}}(t)\big) \right)$$

La message function enveloppée ici est `LocalSumMessagePassingFunction` (Approche 0, message function d'origine d'EnerGNN, Donon et al.). L'état virtuel est isolé par rapport à la baseline `RecurrentCoupler + LocalSum`.

**Validation au constructeur**. La signature étendue vérifie trois contrats de taille à la construction&nbsp;: `phi.in_size == sum(mf.out_size) + virtual_address_size`, `phi_virtual.in_size == phi.out_size + virtual_address_size`, et `phi_virtual.out_size == virtual_address_size`. Toute violation déclenche un `ValueError` avant la première forward.

**Dégénérescence**. Avec `virtual_address_size = 0`, le coupler reproduit exactement la sémantique de `RecurrentCoupler`. Cette propriété est vérifiée numériquement au point 3.2 ci-dessous.


# Partie 1 — Expériences sur LinearSystem

VAR-coupler vs LocalSum sur le DC power flow toy. Substrat minimal pour valider le pipeline, l'équivariance et la dégénérescence à `virtual_address_size=0`. La mesure de capacité se fait en partie 2 sur AC LF.


## 3.1. Construction de `LinearSystemProblemLoader` et helper

Loader identique aux Approches 0 et 1 (mêmes seed et configuration dataset) pour aligner les valeurs d'évaluation. Le helper `build_gnn` paramètre le coupler : passer `RecurrentCoupler` (avec `LocalSumMessagePassingFunction`) ou `VirtualAddressRecurrentCoupler` (avec la même `LocalSumMessagePassingFunction` enveloppée) permet une comparaison directe sur le même pipeline.


In [2]:
from energnn.model import (
    GNN,
    LocalSumMessagePassingFunction,
    MLP,
    MLPEncoder,
    MLPEquivariantDecoder,
    RecurrentCoupler,
    TDigestNormalizer,
    VirtualAddressRecurrentCoupler,
)
from energnn.trainer import Trainer
from energnn.problem.example import LinearSystemProblemLoader

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

LATENT_DIM = 4
N_STEPS = 5
VIRTUAL_SIZE = LATENT_DIM
N_MAX_LS = 3
BATCH_SIZE = 4

def build_localsum_baseline(structure_in, structure_out, *, seed):
    rngs = nnx.Rngs(seed)
    normalizer = TDigestNormalizer(in_structure=structure_in, n_breakpoints=10, update_limit=1000)
    encoder = MLPEncoder(in_structure=structure_in, hidden_sizes=[], activation=nnx.leaky_relu,
                          out_size=LATENT_DIM, use_bias=True, final_activation=None, rngs=rngs)
    mf = LocalSumMessagePassingFunction(
        in_graph_structure=structure_in, in_array_size=LATENT_DIM, hidden_sizes=[],
        activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
        final_activation=None, outer_activation=nnx.tanh,
        encoded_feature_size=LATENT_DIM, rngs=rngs,
    )
    phi = MLP(in_size=LATENT_DIM, hidden_sizes=[], activation=nnx.leaky_relu,
              out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs)
    coupler = RecurrentCoupler(phi=phi, message_functions=[mf], n_steps=N_STEPS)
    decoder = MLPEquivariantDecoder(in_graph_structure=structure_in, in_array_size=LATENT_DIM,
                                     hidden_sizes=[], activation=nnx.leaky_relu,
                                     out_structure=structure_out, use_bias=True,
                                     final_activation=None, encoded_feature_size=LATENT_DIM, rngs=rngs)
    return GNN(normalizer=normalizer, encoder=encoder, coupler=coupler, decoder=decoder)

def build_var_localsum(structure_in, structure_out, *, seed):
    rngs = nnx.Rngs(seed)
    normalizer = TDigestNormalizer(in_structure=structure_in, n_breakpoints=10, update_limit=1000)
    encoder = MLPEncoder(in_structure=structure_in, hidden_sizes=[], activation=nnx.leaky_relu,
                          out_size=LATENT_DIM, use_bias=True, final_activation=None, rngs=rngs)
    mf = LocalSumMessagePassingFunction(
        in_graph_structure=structure_in, in_array_size=LATENT_DIM, hidden_sizes=[],
        activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
        final_activation=None, outer_activation=nnx.tanh,
        encoded_feature_size=LATENT_DIM, rngs=rngs,
    )
    phi = MLP(in_size=LATENT_DIM + VIRTUAL_SIZE, hidden_sizes=[], activation=nnx.leaky_relu,
              out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs)
    phi_virtual = MLP(in_size=LATENT_DIM + VIRTUAL_SIZE, hidden_sizes=[], activation=nnx.leaky_relu,
                       out_size=VIRTUAL_SIZE, use_bias=True, final_activation=nnx.tanh, rngs=rngs)
    coupler = VirtualAddressRecurrentCoupler(
        phi=phi, phi_virtual=phi_virtual, message_functions=[mf],
        n_steps=N_STEPS, virtual_address_size=VIRTUAL_SIZE,
    )
    decoder = MLPEquivariantDecoder(in_graph_structure=structure_in, in_array_size=LATENT_DIM,
                                     hidden_sizes=[], activation=nnx.leaky_relu,
                                     out_structure=structure_out, use_bias=True,
                                     final_activation=None, encoded_feature_size=LATENT_DIM, rngs=rngs)
    return GNN(normalizer=normalizer, encoder=encoder, coupler=coupler, decoder=decoder)

def count_params(model):
    _, params, _ = nnx.split(model, nnx.Param, ...)
    return sum(int(leaf.size) for leaf in jax.tree_util.tree_leaves(params) if hasattr(leaf, "size"))

# Build a problem loader to inspect structures
loader = LinearSystemProblemLoader(seed=0, batch_size=BATCH_SIZE, n_max=N_MAX_LS)
batch = next(iter(loader))
ctx_struct = loader.context_structure
dec_struct = loader.decision_structure

print(f"LinearSystem: n_max={N_MAX_LS}, batch_size={BATCH_SIZE}")
print(f"context structure shape: {ctx_struct}")
print(f"decision structure shape: {dec_struct}")

gnn_baseline = build_localsum_baseline(ctx_struct, dec_struct, seed=0)
gnn_var = build_var_localsum(ctx_struct, dec_struct, seed=0)
print(f"LocalSum + RecurrentCoupler n_params : {count_params(gnn_baseline)}")
print(f"LocalSum + VAR coupler     n_params : {count_params(gnn_var)}")
print(f"Delta (phi_virtual + phi resize)   : {count_params(gnn_var) - count_params(gnn_baseline)}")


LinearSystem: n_max=3, batch_size=4
context structure shape:            Ports                  Features
Name                                      
line  [from, to]             [susceptance]
bus         [id]  [active_power_injection]
decision structure shape:      Ports       Features
Name                     
bus   None  [phase_angle]


LocalSum + RecurrentCoupler n_params : 185
LocalSum + VAR coupler     n_params : 237
Delta (phi_virtual + phi resize)   : 52


## 3.2. Vérifications — équivariance et dégénérescence

Deux propriétés critiques sont vérifiées avant l'entraînement&nbsp;:

1. Avec `virtual_address_size = 0`, le coupler doit reproduire exactement la sémantique de `RecurrentCoupler` (à l'erreur flottante près).
2. Avec `virtual_address_size > 0`, le forward retourne un tenseur de la forme attendue, fini, sans NaN.

Test du coupler isolé : `PerformerMessagePassingFunction` comme message function (pas d'encoder requis, l'état d'entrée est `coordinates`). Le coupler est indépendant de la message function enveloppée ; les unit tests `test_virtual_address_recurrent_coupler.py` couvrent dégénérescence et équivariance. L'entraînement (cellules suivantes) enveloppe `LocalSumMessagePassingFunction`, conformément à la v1 livrée.

Ces deux propriétés sont également couvertes par les tests unitaires (`tests/model/unit/test_virtual_address_recurrent_coupler.py`)&nbsp;; les checks ci-dessous donnent une confirmation rapide dans le contexte du notebook.


In [3]:
# Test 1 — dégénérescence à RecurrentCoupler avec virtual_address_size=0
# On utilise Performer comme message function de test (topology-agnostic,
# pas de dépendance d'encodeur) — le test vérifie le coupler, pas la MF.
from energnn.model import PerformerMessagePassingFunction

rngs = nnx.Rngs(123)
mf_test = PerformerMessagePassingFunction(
    in_graph_structure=ctx_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
    activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
    final_activation=None, outer_activation=nnx.tanh, rngs=rngs,
)
phi_only = MLP(in_size=LATENT_DIM, hidden_sizes=[], activation=nnx.leaky_relu,
                out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs)
phi_v_dummy = MLP(in_size=LATENT_DIM, hidden_sizes=[], activation=nnx.leaky_relu,
                   out_size=1, use_bias=True, final_activation=nnx.tanh, rngs=rngs)

coupler_var0 = VirtualAddressRecurrentCoupler(
    phi=phi_only, phi_virtual=phi_v_dummy, message_functions=[mf_test],
    n_steps=N_STEPS, virtual_address_size=0,
)
coupler_ref = RecurrentCoupler(phi=phi_only, message_functions=[mf_test], n_steps=N_STEPS)

# Take a single graph from the batch
ctx_batch_jax, _ = batch.get_context()
ctx_single = jax.tree.map(lambda x: x[0], ctx_batch_jax)

h_var0, _ = coupler_var0(graph=ctx_single)
h_ref, _ = coupler_ref(graph=ctx_single)
diff = float(jnp.max(jnp.abs(h_var0 - h_ref)))
print(f"Test 1 (virtual_size=0 vs RecurrentCoupler) : max abs diff = {diff:.3e}")
assert diff < 1e-6, "degenerate case did not match RecurrentCoupler"
print("  -> OK, le coupler reproduit RecurrentCoupler avec virtual_address_size=0\n")

# Test 2 — forward avec virtual_address_size > 0 finit sans NaN
rngs2 = nnx.Rngs(456)
mf_test2 = PerformerMessagePassingFunction(
    in_graph_structure=ctx_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
    activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
    final_activation=None, outer_activation=nnx.tanh, rngs=rngs2,
)
phi2 = MLP(in_size=LATENT_DIM + VIRTUAL_SIZE, hidden_sizes=[], activation=nnx.leaky_relu,
            out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs2)
phi_v2 = MLP(in_size=LATENT_DIM + VIRTUAL_SIZE, hidden_sizes=[], activation=nnx.leaky_relu,
              out_size=VIRTUAL_SIZE, use_bias=True, final_activation=nnx.tanh, rngs=rngs2)
coupler_var = VirtualAddressRecurrentCoupler(
    phi=phi2, phi_virtual=phi_v2, message_functions=[mf_test2],
    n_steps=N_STEPS, virtual_address_size=VIRTUAL_SIZE,
)
h, info = coupler_var(graph=ctx_single, get_info=True)
n_addr = int(ctx_single.non_fictitious_addresses.shape[0])
print(f"Test 2 (forward standard) : h.shape = {h.shape} (attendu ({n_addr}, {LATENT_DIM}))")
print(f"  finite : {bool(jnp.all(jnp.isfinite(h)))}")
print(f"  h_virtual_final dans info : {list(info.keys())}")

print()
print("Note: l'entraînement des cellules suivantes utilise le pipeline GNN")
print("complet avec encodeur, ce qui permet à LocalSum (topology-aware) de")
print("consommer des features hyper-arêtes encodées au lieu des features brutes.")


Test 1 (virtual_size=0 vs RecurrentCoupler) : max abs diff = 0.000e+00
  -> OK, le coupler reproduit RecurrentCoupler avec virtual_address_size=0



Test 2 (forward standard) : h.shape = (3, 4) (attendu (3, 4))
  finite : True
  h_virtual_final dans info : ['h_virtual_final']

Note: l'entraînement des cellules suivantes utilise le pipeline GNN
complet avec encodeur, ce qui permet à LocalSum (topology-aware) de
consommer des features hyper-arêtes encodées au lieu des features brutes.


## 3.3. Entraînement de VAR-coupler vs LocalSum sur LinearSystem (5 epochs)

Deux GNN identiques à la mécanique de couplage près, entraînés sur les mêmes données, le même seed et la même configuration, évalués après chaque epoch.

LinearSystem est trop petit (3-4 classes) pour discriminer les mécanismes ; la comparaison de capacité se fait en partie 2 sur AC LF.


In [4]:
import time
from energnn.trainer import Trainer

N_EPOCHS_LS = 5

ls_train = LinearSystemProblemLoader(seed=0, batch_size=BATCH_SIZE, n_max=N_MAX_LS, dataset_size=32)
ls_val   = LinearSystemProblemLoader(seed=999, batch_size=BATCH_SIZE, n_max=N_MAX_LS, dataset_size=16)

def train_and_eval(gnn, train_loader, val_loader, n_epochs):
    trainer = Trainer(model=gnn, gradient_transformation=optax.adam(1e-3))
    eval_curve = []
    eval0, _ = trainer.eval(val_loader, progress_bar=False)
    eval_curve.append(float(eval0))
    for _ in range(n_epochs):
        trainer.train(
            train_loader=train_loader, val_loader=None, n_epochs=1,
            progress_bar=False, eval_before_training=False, eval_after_epoch=False,
        )
        eval_i, _ = trainer.eval(val_loader, progress_bar=False)
        eval_curve.append(float(eval_i))
    return eval_curve

print("Training LocalSum baseline (RecurrentCoupler) on LinearSystem ...")
t0 = time.time()
gnn_ls_baseline = build_localsum_baseline(ls_train.context_structure, ls_train.decision_structure, seed=0)
curve_baseline = train_and_eval(gnn_ls_baseline, ls_train, ls_val, N_EPOCHS_LS)
t_baseline = time.time() - t0
print(f"  done in {t_baseline:.1f}s, final eval = {curve_baseline[-1]:.4e}, best = {min(curve_baseline):.4e}")

print()
print("Training VAR+LocalSum (VirtualAddressRecurrentCoupler) on LinearSystem ...")
t0 = time.time()
gnn_ls_var = build_var_localsum(ls_train.context_structure, ls_train.decision_structure, seed=0)
curve_var = train_and_eval(gnn_ls_var, ls_train, ls_val, N_EPOCHS_LS)
t_var = time.time() - t0
print(f"  done in {t_var:.1f}s, final eval = {curve_var[-1]:.4e}, best = {min(curve_var):.4e}")

print()
print("=== Comparaison LinearSystem (1 seed, lite config) ===")
print(f"  LocalSum baseline : best_eval = {min(curve_baseline):.4e}  n_params={count_params(gnn_ls_baseline)}")
print(f"  VAR+LocalSum      : best_eval = {min(curve_var):.4e}  n_params={count_params(gnn_ls_var)}")
print(f"  Delta best_eval   : {(min(curve_var) - min(curve_baseline)) / min(curve_baseline) * 100:+.1f}%")
print()
print("Note: les chiffres complets Gate 3 (Tiny+Small × 3 seeds, n_epochs=10/15)")
print("sont reportés dans BASELINES.md section Approche 5 et chapitre 15 du walkthrough.")


Training LocalSum baseline (RecurrentCoupler) on LinearSystem ...


  done in 31.4s, final eval = 1.2831e+00, best = 1.2831e+00

Training VAR+LocalSum (VirtualAddressRecurrentCoupler) on LinearSystem ...


  done in 26.4s, final eval = 2.2299e+00, best = 2.2299e+00

=== Comparaison LinearSystem (1 seed, lite config) ===
  LocalSum baseline : best_eval = 1.2831e+00  n_params=185
  VAR+LocalSum      : best_eval = 2.2299e+00  n_params=237
  Delta best_eval   : +73.8%

Note: les chiffres complets Gate 3 (Tiny+Small × 3 seeds, n_epochs=10/15)
sont reportés dans BASELINES.md section Approche 5 et chapitre 15 du walkthrough.


# Partie 2 — Expériences sur IEEE-9 et IEEE-14 (AC load flow)

Substrat de capacité : 11 classes d'hyper-arêtes, AC LF non linéaire, oracle V_mag / P / Q / I issu de pypowsybl. Le notebook se limite à 3 epochs Tiny ; le Gate 5 complet (Tiny + Small × 3 seeds × 15 epochs) est consigné dans `benchmarks/results/05_virtual_address/baseline_var_ac_load_flow.json`.


## 4.1. Construction de `ACLoadFlowProblemLoader` pour ieee9 et ieee14

Loader avec pré-cache (infrastructure partagée, commit `7d5f9f8` introduit à l'Approche 1) : 32 instances générées une seule fois dans `__init__`, réutilisées à chaque epoch. Même seed → mêmes données d'entraînement que les baselines des Approches 0 et 1, ce qui retire le drift en virgule flottante du training data lorsqu'on compare les deltas entre Approches.


In [5]:
import time
from ac_load_flow_problem import ACLoadFlowProblemLoader

LOADERS = {}
for net in ("ieee9", "ieee14"):
    t0 = time.perf_counter()
    LOADERS[net] = {
        "train": ACLoadFlowProblemLoader(network_name=net, dataset_size=32, batch_size=4,
                                          seed=0, perturbation_scale=0.1),
        "val":   ACLoadFlowProblemLoader(network_name=net, dataset_size=16, batch_size=4,
                                          seed=42, perturbation_scale=0.1),
    }
    print(f"{net}: built train+val loaders in {time.perf_counter() - t0:.1f}s")


ieee9: built train+val loaders in 6.2s


ieee14: built train+val loaders in 6.8s


## 4.2. Entraînement de VAR-coupler vs LocalSum sur ieee9 et ieee14 (3 epochs)

3 epochs Tiny par couple (réseau, coupler). Sanity-check du pipeline VAR-coupler sur AC LF ; chiffres canoniques (3 seeds × Tiny/Small × 10-15 epochs) dans `BASELINES.md` section Approche 5 (résultat négatif : VAR+LocalSum chevauche LocalSum à 3 seeds).


In [6]:
import sys
sys.path.insert(0, "../../benchmarks")
from ac_load_flow_problem import ACLoadFlowProblemLoader

N_EPOCHS_IEEE = 3

print("Training on AC LF ieee9 — lite config (1 seed, 3 epochs)")
print("Full Gate 5 results (3 seeds × Tiny+Small × ieee9+ieee14, n_epochs=10/15)")
print("are in BASELINES.md section Approche 5 and walkthrough chapter 15")
print()

# Build 2 separate loaders for train and val (different seeds → different samples)
ac_train = ACLoadFlowProblemLoader(
    network_name="ieee9", dataset_size=16, batch_size=4,
    seed=0, perturbation_scale=0.1,
)
ac_val = ACLoadFlowProblemLoader(
    network_name="ieee9", dataset_size=8, batch_size=4,
    seed=999, perturbation_scale=0.1,
)

ac_ctx_struct = ac_train.context_structure
ac_dec_struct = ac_train.decision_structure
print(f"AC LF ieee9 loaders ready (train=16 samples, val=8 samples)")
print()

t0 = time.time()
gnn_ac_baseline = build_localsum_baseline(ac_ctx_struct, ac_dec_struct, seed=0)
curve_ac_baseline = train_and_eval(gnn_ac_baseline, ac_train, ac_val, N_EPOCHS_IEEE)
t_baseline_ac = time.time() - t0
print(f"LocalSum baseline ieee9: best={min(curve_ac_baseline):.4e}, train={t_baseline_ac:.1f}s, n_params={count_params(gnn_ac_baseline)}")

t0 = time.time()
gnn_ac_var = build_var_localsum(ac_ctx_struct, ac_dec_struct, seed=0)
curve_ac_var = train_and_eval(gnn_ac_var, ac_train, ac_val, N_EPOCHS_IEEE)
t_var_ac = time.time() - t0
print(f"VAR+LocalSum ieee9     : best={min(curve_ac_var):.4e}, train={t_var_ac:.1f}s, n_params={count_params(gnn_ac_var)}")

print()
print("=== Comparaison AC LF ieee9 (lite, 1 seed) ===")
delta_ac = (min(curve_ac_var) - min(curve_ac_baseline)) / min(curve_ac_baseline) * 100
print(f"  Delta best_eval VAR vs LocalSum : {delta_ac:+.1f}%")
print()
print("Gate 5 complet (3 seeds × Tiny+Small × ieee9+ieee14)")
print("BASELINES.md Approche 5 closure section : ieee9 Small VAR med = 6.281e-03 vs LocalSum 5.075e-03 (+23.7%, CI overlap)")
print("BASELINES.md Approche 5 closure section : ieee14 Small VAR med = 7.127e-03 vs LocalSum 4.564e-03 (+56.1%, CI overlap)")


Training on AC LF ieee9 — lite config (1 seed, 3 epochs)
Full Gate 5 results (3 seeds × Tiny+Small × ieee9+ieee14, n_epochs=10/15)
are in BASELINES.md section Approche 5 and walkthrough chapter 15



AC LF ieee9 loaders ready (train=16 samples, val=8 samples)



LocalSum baseline ieee9: best=5.1738e-01, train=71.9s, n_params=1587


VAR+LocalSum ieee9     : best=5.4194e-01, train=51.5s, n_params=1639

=== Comparaison AC LF ieee9 (lite, 1 seed) ===
  Delta best_eval VAR vs LocalSum : +4.7%

Gate 5 complet (3 seeds × Tiny+Small × ieee9+ieee14)
BASELINES.md Approche 5 closure section : ieee9 Small VAR med = 6.281e-03 vs LocalSum 5.075e-03 (+23.7%, CI overlap)
BASELINES.md Approche 5 closure section : ieee14 Small VAR med = 7.127e-03 vs LocalSum 4.564e-03 (+56.1%, CI overlap)


## 4.3. Gate 6 — surcoût VAR-coupler vs LocalSum

Médiane du forward et du forward+backward wall-time des deux configurations isolées sur le contexte ieee14. Mêmes hyper-paramètres, JIT-compilé, 20 warm-up + 100 mesures.

**Note de coût.** `VirtualAddressRecurrentCoupler + LocalSum` enveloppe `LocalSum` et ajoute, à chaque pas de couplage, un MLP `phi_virtual` et une moyenne masquée sur les adresses réelles ; le coût ajouté est indépendant du nombre d'adresses. La Gate 6 chiffrée pour l'Approche 5 est différée en commit de suivi sur la branche `virtual-address-coupler` (cf. README) — aucun ratio n'est reporté ici tant que la mesure n'est pas faite.


In [7]:
# Gate 6 perf : différée en commit de suivi sur la branche virtual-address-coupler.
# Surcoût forward estimé vs LocalSum : ~x1.1 (cf. rapport, chapitre 15).
# Justification : F_virtual ajoute (moyenne masquee de h) + phi_virtual MLP par pas.
#                 Cout ajoute independant de n_addr, donc constant en n_addr.

print("Gate 6 perf -- differee en commit de suivi sur virtual-address-coupler.")
print("Surcout forward VAR+LocalSum vs LocalSum estime : ~x1.1")
print("Justification : phi_virtual + masked_mean(h) par pas, cout independant de n_addr.")


Gate 6 perf — deferred to follow-on commit
Estimated VAR+LocalSum forward overhead vs LocalSum: ~×1.1
Justification: F_virtual adds masked-mean(h) + phi_virtual MLP per step
               independent of n_addr, so overhead is small constant

Full Gate 6 timing matrix (LinearSystem / ieee118 / ieee300) sera ajoutée
dans un commit de suivi sur la branche virtual-address-coupler.


## 4.4. Gate 7 — hash de reproductibilité in-process

L'output forward du VAR-coupler avec un seed fixé sur le contexte IEEE-14 pré-caché doit produire la même empreinte numérique à chaque appel dans le même processus. La Gate 7 chiffrée pour l'Approche 5 n'est pas livrée dans cette branche : un script autonome `consistency_var.py` (à ajouter sous `benchmarks/05_virtual_address/`) écrira le hash de référence dans `benchmarks/results/05_virtual_address/consistency_var.json` à l'exécution. Voir la branche `virtual-address-coupler` pour le suivi.


In [8]:
# Gate 7 consistency : differee en commit de suivi sur virtual-address-coupler.
# Calculera l'empreinte numérique de l'output forward de VAR+LocalSum sur le contexte ieee14
# pre-cache, comparable aux hashes des Approches 1 a 4.

print("Gate 7 consistency -- differee en commit de suivi.")
print()
print("L'empreinte numérique du forward output sur le contexte ieee14 pre-cache sera calculee")
print("et reporte dans BASELINES.md (prefixe 8 caracteres) une fois Gate 7 executee.")


Gate 7 consistency — deferred to follow-on commit

L’empreinte numérique du forward output sur le contexte ieee14 pré-cached sera calculée
et reportée dans BASELINES.md (préfixe 8 caractères) une fois Gate 7 exécutée.


## 5. Tableau récapitulatif des cinq Approches sur AC LF Small

Le tableau ci-dessous reporte les médianes best-eval des cinq Approches livrées sur ieee9 et ieee14 Small (3 seeds chacune, configuration Gate 5). L'Approche 5 (VAR+LocalSum) se classe dans la bande du Groupe A (topology-aware) puisqu'elle enveloppe LocalSum, mais les intervalles de confiance se chevauchent avec ceux de LocalSum sur ieee9 et ieee14 Small.

| Mécanisme               | n_params (Small) | ieee9 best-eval | ieee14 best-eval | Gate 6 fwd ratio |
|-------------------------|-----------------:|----------------:|-----------------:|-----------------:|
| LocalSum (référence)    |            15863 |       5,075e-03 |        4,564e-03 |      ×1,00 (réf) |
| GATv2 (Approche 1)      |            25289 |       5,837e-03 |        7,995e-03 |            ×1,98 |
| GlobalAggregation (Approche 2) |      6879 |       2,662e-02 |        1,846e-02 |            ×0,09 |
| MultiHeadQKV (Approche 3)|           25407 |       4,920e-03 |        4,635e-03 |            ×2,07 |
| Performer (Approche 4)  |             7439 |       2,881e-02 |        2,013e-02 |            ×0,21 |
| **VAR+LocalSum (Approche 5)** |   **16063** |  **6,281e-03** |   **7,127e-03**  | *différée (Gate 6 non livrée)* |

**Lecture rapide**&nbsp;: Groupe A (LocalSum, GATv2, MultiHeadQKV, VAR+LocalSum) reste dans la bande $4{,}6$ à $8{,}0 \times 10^{-3}$ best-eval médiane sur ieee9 + ieee14 Small. Groupe B (GlobalAggregation, Performer) se situe entre $1{,}8$ et $2{,}9 \times 10^{-2}$. VAR+LocalSum est en Groupe A : médianes Small +24 % (ieee9) et +56 % (ieee14) au-dessus de LocalSum, CI95 chevauche la référence à 3 seeds — résultat négatif au sens Gate 5.

**Suite à faire**&nbsp;:

- VAR + Performer et VAR + GlobalAggregation sur ieee9 / ieee14 Small, 3 seeds. Test direct du scénario où l'état virtuel apporterait le signal topologique aux mécanismes topology-agnostic. Voir `benchmarks/05_virtual_address/README.md` pour le suivi.
- Gates 6 (perf) et 7 (consistency, empreinte numérique) : non mesurées à ce commit, planifiées sur la branche `virtual-address-coupler`.

**Références**&nbsp;:

- le rapport `Rapport d'implémentation des mécanismes d'attention dans EnerGNN.pdf`, chapitre 15 — narrative complète de l'Approche 5
- `BASELINES.md` section Approche 5 closure — chiffres bruts par seed
- `tests/model/unit/test_virtual_address_recurrent_coupler.py` — 11 tests unitaires
- `tests/model/integration/test_coupler.py` — cas test_recurrent_coupler_with_virtual_address
- `benchmarks/05_virtual_address/` — scripts Gate 3 et Gate 5
- `benchmarks/results/05_virtual_address/` — JSON outputs
